# 01 · Set up the telemetry Eventhouse

Creates the evaluation tables in a KQL database, their JSON ingestion mappings, OneLake availability for the
tables the app reads, and KQL functions for analysis. Then shortcuts the app tables into a lakehouse so the
Fabric app can read them through the SQL analytics endpoint.

**Before you start**
1. An Eventhouse with a KQL database (for example `copilot_eval`). Copy its **Query URI**.
2. A lakehouse for the app (for example `eval_lakehouse`).
3. Workspace monitoring turned on for the workspace that holds the semantic models (this is where the DAX
   comes from; it creates its own monitoring Eventhouse).

Optional UI-configured notebook path. Default acceptance uses the local setup/rehearsal scripts.
No workspace creation/deletion, admin/security changes or automatic jobs. Apply/publish flags default false.

In [ ]:
# Make the cfeval package available.
# - Local clone (VS Code / Jupyter): the cell below finds src/ in the repo.
# - Fabric notebook: uncomment the %pip line (or attach an Environment with the package).
# %pip install -q "cfeval[kusto] @ git+https://github.com/bgbailey/copilot_iq_eval"
import sys, pathlib
repo = pathlib.Path.cwd()
while repo != repo.parent and not (repo / "src" / "cfeval").exists():
    repo = repo.parent
if (repo / "src" / "cfeval").exists() and str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))
import cfeval
print("cfeval", cfeval.__version__)

In [ ]:
import os

apply_setup = False
publish_sample_history = False

os.environ.update({
    "TENANT_ID": "<tenant id>",
    "TENANT_LABEL": "<tenant name / domain>",
    "OPERATOR_UPN": "eval.operator@<your-domain>",
    "EVENTHOUSE_QUERY_URI": "https://<your-eventhouse>.z0.kusto.fabric.microsoft.com",  # Eventhouse > Query URI
    "EVENTHOUSE_DATABASE": "<actual engine databaseName, often the KQL database GUID>",
})
from cfeval.config import Settings
settings = Settings.load(env_file=None)

## Review the setup script
The same script is in `kql/setup.kql` if you prefer to run it in a KQL queryset.

In [ ]:
from cfeval.kqlgen import setup_script
print(setup_script(settings.eventhouse_database))

## Apply it
Commands after a `// optional` line are best effort (for example streaming ingestion).

In [ ]:
from cfeval.cli import main
if apply_setup:
    if main(["--env", "", "setup-eventhouse"]):
        raise RuntimeError("Setup failed; stop before shortcuts or sample publication")
else:
    print("Setup not applied (apply_setup=False).")

## Shortcut the app tables into the lakehouse
OneLake availability writes each app table independently; five minutes is a minimum setting, not a deadline. A shortcut makes it
queryable in the lakehouse SQL analytics endpoint, which the Fabric app's connector reads.
Find the IDs in each item's URL: `.../groups/<workspace id>/lakehouses/<lakehouse id>` and
`.../groups/<workspace id>/databases/<kql database id>`.

In [ ]:
if apply_setup:
    if main(["--env", "", "shortcuts",
             "--lakehouse-workspace", "<workspace id>", "--lakehouse", "<lakehouse ITEM id>",
             "--kql-workspace", "<workspace id>", "--kql-database", "<KQL DATABASE item GUID>"]):
        raise RuntimeError("Shortcut setup failed; existing targets were not overwritten")

## Optional: load simulated history for the app
Eight simulated runs over two weeks, labelled `simulated` so they're easy to filter out.

In [ ]:
if publish_sample_history:
    main(["--env", "", "demo-data", "--runs", "8", "--days", "14", "--publish"])
else:
    print("No simulated history published (publish_sample_history=False).")

## Check

In [ ]:
from cfeval.kusto import KustoQueryClient

if apply_setup:
    kc = KustoQueryClient(settings.eventhouse_query_uri, settings=settings)
    print(kc.mgmt(settings.eventhouse_database, ".show tables"))
else:
    print("No cloud readback: setup was not applied.")